In [1]:

from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

batch_size = 15
train_loader = train_loader(batch_size=batch_size)

class att(torch.nn.Module):
    def __init__(self):
        super().__init__()


        self.pattn1 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn2 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn3 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn4 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn5 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn6 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn7 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn8 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn9 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn10 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        
        self.mlp = tg.nn.models.MLP(
            in_channels=10,
            hidden_channels=5,
            num_layers=3,
            out_channels=1,
        )

    def forward(self, data) -> torch.Tensor:
        batch, x  = data.batch, data.x
        h, mask = utils.to_dense_batch(x=x, batch=batch)
        h = self.pattn1(h).relu()
        h = self.pattn2(h).relu()
        h = self.pattn3(h).relu()
        h = self.pattn4(h).relu()
        h = self.pattn5(h).relu()
        h = self.pattn6(h).relu()
        h = self.pattn7(h).relu()
        h = self.pattn8(h).relu()
        h = self.pattn9(h).relu()
        h = self.pattn10(h)[mask]
        h = self.mlp(h)

        return h
    

class mlp(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = tg.nn.models.MLP(
            in_channels=10,
            hidden_channels=256,
            num_layers=10,
            out_channels=1,
        )

    def forward(self, data) -> torch.Tensor:
        x = data.x
        h = self.mlp(x)

        return h
    
class mix(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.mlp1 = tg.nn.models.MLP(
            in_channels=10,
            hidden_channels=256,
            num_layers=6,
            out_channels=5,
        )
        self.conv1 = tg.nn.conv.GATv2Conv(5, 5)
        self.mlp2 = tg.nn.models.MLP(
            in_channels=5,
            hidden_channels=256,
            num_layers=6,
            out_channels=4,
        )
        self.pattn = tg.nn.attention.PerformerAttention(channels=4, heads=1)
        self.mlp3 = tg.nn.models.MLP(
            in_channels=4,
            hidden_channels=256,
            num_layers=6,
            out_channels=1,
        )

    def forward(self, data) -> torch.Tensor:
        batch, x, edge_index  = data.batch, data.x, data.edge_index
        x = self.mlp1(x)
        x = self.conv1(x=x, edge_index = edge_index)
        x = self.mlp2(x)
        x, mask = utils.to_dense_batch(x=x, batch=batch)
        x = self.pattn10(x).relu()[mask]
        x = self.mlp3(x)

        return x
    
model = att()
model = mlp()

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.9, patience=50)

In [2]:
curr_loss = 0.024

In [8]:

from Training_utils import train_loader


In [6]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=100)

In [9]:
batch_size = 1
train_loader = train_loader(batch_size=batch_size)

In [3]:
model.load_state_dict(torch.load("data/models/lMLP_supervised.pth")['model_state'])

<All keys matched successfully>

In [10]:

for i in range(10000):
    loss = train(model=model, loader=train_loader, optimizer=optimizer, device='cpu')
    if loss > curr_loss:
        print(f"iteration: {i}, loss: {loss}, model loss: {curr_loss}:")
    elif loss <= curr_loss:
        curr_loss = loss
        print(f"iteration: {i}, new model loss: {loss}")
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/lMLP_supervised.pth")

    scheduler.step(loss)

iteration: 0, loss: 0.6719714630691503, model loss: 0.024:
iteration: 1, loss: 0.5778528076215562, model loss: 0.024:
iteration: 2, loss: 0.5352447859794748, model loss: 0.024:
iteration: 3, loss: 0.5002897996455431, model loss: 0.024:
iteration: 4, loss: 0.47211572631600907, model loss: 0.024:
iteration: 5, loss: 0.4547606078241105, model loss: 0.024:
iteration: 6, loss: 0.42999196028209885, model loss: 0.024:
iteration: 7, loss: 0.41436445208224043, model loss: 0.024:
iteration: 8, loss: 0.3969704868081449, model loss: 0.024:
iteration: 9, loss: 0.383952224616554, model loss: 0.024:
iteration: 10, loss: 0.36562878187661146, model loss: 0.024:
iteration: 11, loss: 0.35710187479641964, model loss: 0.024:
iteration: 12, loss: 0.35600875873668963, model loss: 0.024:
iteration: 13, loss: 0.35244863071817567, model loss: 0.024:
iteration: 14, loss: 0.3315600262139924, model loss: 0.024:
iteration: 15, loss: 0.329273922665214, model loss: 0.024:
iteration: 16, loss: 0.312565086980943, model